# HOF-Net vs LGBM(운영 v3) 비교

같은 변수(v3 15개)·같은 split·같은 지표. 실행 결과는 `outputs/hofnet/` (run_log.txt, csv, summary.json).
Jupyter 커널 연결 문제로 `python hofnet_experiment.py` 로 실행했다 (코드 동일).

In [ ]:
"""
HOF-Net vs LGBM(운영 v3) 비교 실험 — 같은 변수·표본·split·지표 (stuff_pipeline_261010_hofnet.ipynb 와 같은 코드)

실행 : python hofnet_experiment.py [--smoke]      (--smoke : 표본 2만 구, 2 epoch 로 코드 점검)
입력 : outputs/vaa_exp/cache/ (3차 실험 캐시), models/v3 (운영 LGBM)
출력 : outputs/hofnet/ (비교표 csv, 학습 기록, 요약 json, 모델 state_dict(.pt, git 제외))
"""
import json, os, sys, time, gc, warnings
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score

import stuff_pipeline as sp
import stuff_model as sm
import export_web_data as ew
import hof_net as hn

warnings.filterwarnings('ignore')
SMOKE = '--smoke' in sys.argv
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.set_num_threads(min(12, os.cpu_count()))
OUT = os.path.join('outputs', 'hofnet')
CACHE = os.path.join('outputs', 'vaa_exp', 'cache')
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
log = lambda *a: print(f'[{time.time() - T0:6.0f}s]', *a, flush=True)
num = lambda s: s.to_numpy('float64', na_value=np.nan)

In [ ]:
# ---------------------------------------------------------------------------
# 1. 데이터 (운영 v3 와 같은 변수 : v3 모델 객체가 만드는 피처 15개)
# ---------------------------------------------------------------------------
v3 = sm.load_model(os.path.join('models', 'v3'))
v3_scaler = joblib_load = __import__('joblib').load(os.path.join('models', 'v3', 'stage4_stuff_scaler.joblib'))
FEATS = v3.features
NUM = [f for f in FEATS if f not in sp.STUFF_CAT_FEATURES]
CAT = sp.STUFF_CAT_FEATURES
WEB = ['pa_event', 'is_swing', 'is_whiff', 'result', 'xwoba', 'plate_z_norm']
cols = sorted(set(v3.input_features) | {'pitcher', 'game_date', 'description', 'type', 'estimated_woba_using_speedangle', 'zone',
                                        'plate_x', 'waste'} | set(WEB))
fr = {}
for y in (2024, 2025, 2026):
    fr[y] = pd.read_parquet(os.path.join(CACHE, f'frame_{y}.parquet'), columns=cols + (['y3', 'split'] if y == 2025 else []))
    Xm = v3.model_frame(fr[y][v3.input_features])
    fr[y]['vaa_aa'] = Xm['vaa_aa'].to_numpy()
valid_start = pd.Timestamp(pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start'])
d25 = fr[2025]
is_tr = (d25['game_date'] < valid_start).to_numpy()
samp = d25[d25['y3'].notna()]
tr_rows, es_rows, va_rows = samp[samp['split'] == 'tr'], samp[samp['split'] == 'es'], samp[samp['split'] == 'va']
if SMOKE:
    tr_rows, es_rows, va_rows = tr_rows.sample(20000, random_state=SEED), es_rows.sample(4000, random_state=SEED), va_rows.sample(4000, random_state=SEED)
log('표본 train', len(tr_rows), 'holdout', len(es_rows), 'valid', len(va_rows))

prep = hn.Prep(tr_rows, NUM, CAT)
T = lambda df: prep(df[FEATS])
Y = lambda df: torch.tensor(df['y3'].to_numpy(np.float32))
tr_t, es_t, va_t = (*T(tr_rows), Y(tr_rows)), (*T(es_rows), Y(es_rows)), T(va_rows)
y_va = va_rows['y3'].to_numpy()

In [ ]:
# ---------------------------------------------------------------------------
# 2. HOF-Net 학습 : A0 전체, A1 대각 사후분포(R = I), A5 결정적(W = μ)
# ---------------------------------------------------------------------------
VARIANTS = {'A0': dict(cov='full', latent=True), 'A1': dict(cov='diag', latent=True), 'A5': dict(cov='full', latent=False)}
nets, hists, times = {}, {}, {}
for k, kw in VARIANTS.items():
    t = time.time()
    net = hn.HOFNet(len(NUM), prep.cat_cards, **kw)
    log(f'--- HOF-Net {k} {kw}')
    hists[k] = hn.fit(net, tr_t, es_t, epochs=2 if SMOKE else 30, patience=4, log=lambda s: log('  ', s))
    times[k] = time.time() - t
    nets[k] = net
    torch.save({'state': net.state_dict(), 'kw': kw, 'num': NUM, 'cat': CAT, 'mean': prep.mean.to_dict(), 'std': prep.std.to_dict(),
                'cats': prep.cats}, os.path.join(OUT, f'hof_{k}.pt'))
    hists[k].assign(variant=k).to_csv(os.path.join(OUT, f'train_history_{k}.csv'), index=False, encoding='utf-8-sig')
    log(f'  {k} 학습 {times[k]:.0f}초, {len(hists[k])} epoch')

In [ ]:
# ---------------------------------------------------------------------------
# 3. 점 예측 · 확률 예측 (2025 검증)
# ---------------------------------------------------------------------------
def prob_metrics(mean, var, y=y_va):
    sd = np.sqrt(var)
    r = {'RMSE': np.sqrt(mean_squared_error(y, mean)), 'MAE': mean_absolute_error(y, mean), 'R2': r2_score(y, mean),
         'NLL': float(np.mean(0.5 * (np.log(2 * np.pi * var) + (y - mean) ** 2 / var))),
         'CRPS': float(np.mean(hn.gauss_crps(y, mean, sd)))}
    ce = []
    for c in (0.5, 0.8, 0.9, 0.95):
        zq = stats.norm.ppf(0.5 + c / 2)
        cov = float(np.mean(np.abs(y - mean) <= zq * sd))
        r[f'커버리지 {int(c * 100)}%'] = cov
        ce.append(abs(cov - c))
    r['보정 오차 (평균 |커버리지 − 명목|)'] = float(np.mean(ce))
    r['90% 구간 평균 폭'] = float(np.mean(2 * stats.norm.ppf(0.95) * sd))
    return r

fit_rows = pd.concat([tr_rows, es_rows])
lg_fit = v3.predict(fit_rows[v3.input_features])
lg_va = v3.predict(va_rows[v3.input_features])
res = {}
sig = float(np.std(fit_rows['y3'].to_numpy() - lg_fit))
res['LGBM v3 + 상수 분산'] = prob_metrics(lg_va, np.full(len(lg_va), sig ** 2))
params = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=200, random_state=SEED, verbose=-1)
Xfit, Xva = v3.model_frame(fit_rows[v3.input_features]), v3.model_frame(va_rows[v3.input_features])
var_m = lgb.LGBMRegressor(**params).fit(Xfit, np.log((fit_rows['y3'].to_numpy() - lg_fit) ** 2 + 1e-6))
res['LGBM v3 + 이분산 오차 모델'] = prob_metrics(lg_va, np.exp(var_m.predict(Xva)) * 1.27)   # E[log χ²₁] 보정 (e^1.27 ≈ 1/0.28)
q = {a: lgb.LGBMRegressor(objective='quantile', alpha=a, **params).fit(Xfit, fit_rows['y3']).predict(Xva) for a in (0.05, 0.5, 0.95)}
qr = {'RMSE': np.sqrt(mean_squared_error(y_va, q[0.5])), 'MAE': mean_absolute_error(y_va, q[0.5]), 'R2': r2_score(y_va, q[0.5]),
      '커버리지 90%': float(np.mean((y_va >= q[0.05]) & (y_va <= q[0.95]))), '90% 구간 평균 폭': float(np.mean(q[0.95] - q[0.05]))}
res['LGBM 분위 회귀 (5·50·95%)'] = qr
hof_va = {}
for k, net in nets.items():
    p = hn.predict(net, va_t)
    hof_va[k] = p
    res[f'HOF-Net {k}'] = prob_metrics(p['mean'], p['var'] + p['u_lat'])
point_prob = pd.DataFrame(res)
point_prob.to_csv(os.path.join(OUT, 'point_prob.csv'), encoding='utf-8-sig')
log('점·확률 예측\n' + point_prob.round(4).to_string())

dec = pd.DataFrame({'LGBM v3': lg_va, **{f'HOF {k}': hof_va[k]['mean'] for k in nets}, 'actual': y_va})
deciles = pd.DataFrame({m: dec.groupby(pd.qcut(dec[m], 10, labels=False))['actual'].mean() for m in dec.columns if m != 'actual'}).T
deciles.to_csv(os.path.join(OUT, 'deciles.csv'), encoding='utf-8-sig')

In [ ]:
# ---------------------------------------------------------------------------
# 4. 구위 점수 단계 지표 (지금까지와 같은 정의)
# ---------------------------------------------------------------------------
def season_pred(k):
    out = {}
    for y in fr:
        if k == 'LGBM v3':
            out[y] = v3.predict(fr[y][v3.input_features])
        else:
            out[y] = hn.predict(nets[k], T(fr[y]))['mean']
    return out

def within_corr(d, col, loc, min_n=100):
    x = pd.DataFrame({'g1': d['pitcher'].to_numpy(), 'g2': d['pitch_type'].astype(str).to_numpy(), 'a': num(d[col]), 'b': num(d[loc])}).dropna()
    x = x.assign(ab=x['a'] * x['b'], aa=x['a'] ** 2, bb=x['b'] ** 2)
    s = x.groupby(['g1', 'g2']).agg(n=('a', 'size'), a=('a', 'sum'), b=('b', 'sum'), ab=('ab', 'sum'), aa=('aa', 'sum'), bb=('bb', 'sum'))
    s = s[s['n'] >= min_n]
    r = (s['ab'] - s['a'] * s['b'] / s['n']) / np.sqrt((s['aa'] - s['a'] ** 2 / s['n']) * (s['bb'] - s['b'] ** 2 / s['n']))
    ok = r.notna() & np.isfinite(r)
    return float(np.average(r[ok], weights=s['n'][ok]))

def validity(d, col):
    d = d.assign(swing=d['description'].isin(ew.SWING_DESC), whiff=d['description'].isin(sp.WHIFF_DESC), xc=d['estimated_woba_using_speedangle'].where(d['type'] == 'X'))
    pp = d.groupby(['pitcher', 'pitch_type'], observed=True).agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum'), xc=('xc', 'mean')).query('n >= 100')
    pt = d.groupby('pitcher').agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum')).query('n >= 300')
    return {'투수×구종 ρ(구위, whiff%)': stats.spearmanr(pp['s'], pp['wh'] / pp['sw']).statistic,
            '투수×구종 ρ(구위, xwOBAcon)': stats.spearmanr(pp['s'], pp['xc'], nan_policy='omit').statistic,
            '투수 ρ(구위, whiff%)': stats.spearmanr(pt['s'], pt['wh'] / pt['sw']).statistic}

def pt_means(d, col, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str)).groupby(['pitcher', 'pitch_type'])[col].agg(['mean', 'count'])
    return g[g['count'] >= min_n]['mean']

def pt_whiff(d, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str), sw=d['description'].isin(ew.SWING_DESC), wh=d['description'].isin(sp.WHIFF_DESC)) \
         .groupby(['pitcher', 'pitch_type']).agg(n=('sw', 'size'), sw=('sw', 'sum'), wh=('wh', 'sum'))
    g = g[g['n'] >= min_n]
    return g['wh'] / g['sw']

def buckets(d, col):
    bins = pd.cut(d[col], ew.STUFF_BINS, labels=ew.STUFF_BIN_LABELS, right=False)
    return pd.DataFrame({lab: ew.outcome_rates(d[bins == lab]) for lab in ew.STUFF_BIN_LABELS}).T

scalers, stuff_cols, rows = {}, {}, {}
for k in ['LGBM v3'] + list(nets):
    pred = season_pred(k)
    sc = v3_scaler if k == 'LGBM v3' else sp.StuffScaler().fit(pred[2025][is_tr & ~d25['waste'].to_numpy()])
    scalers[k] = sc
    col = f's_{k}'
    for y in fr:
        raw = sc.transform(pred[y]).astype('float32')
        fr[y][f'raw_{k}'] = raw
        fr[y][col] = pd.Series(raw, index=fr[y].index).where(~fr[y]['waste'])
    r = {}
    r.update({f'검증 {a}': v for a, v in validity(fr[2025][~is_tr], col).items()})
    for y in (2024, 2026):
        b = buckets(fr[y], col)
        r[f'{y} BA 격차'] = b.loc['<40', 'ba'] - b.loc['60+', 'ba']
        r[f'{y} Whiff 격차'] = b.loc['60+', 'whiff'] - b.loc['<40', 'whiff']
    for a, b in [(2024, 2025), (2025, 2026)]:
        mm = pd.concat([pt_means(fr[a], col), pt_means(fr[b], col)], axis=1, join='inner')
        r[f'안정성 r(구위 {a}, 구위 {b})'] = mm.corr().iloc[0, 1]
        w = pd.concat([pt_means(fr[a], col), pt_whiff(fr[b])], axis=1, join='inner')
        r[f'예측력 ρ(구위 {a}, whiff% {b})'] = stats.spearmanr(w.iloc[:, 0], w.iloc[:, 1]).statistic
    d26 = fr[2026]
    r['투수×구종 내 corr(점수, 높이 정규화)'] = within_corr(d26, col, 'plate_z_norm')
    r['투수×구종 내 corr(점수, |plate_x|)'] = within_corr(d26.assign(abs_x=d26['plate_x'].abs()), col, 'abs_x')
    rawc = d26[f'raw_{k}']
    top = rawc >= rawc.quantile(0.99)
    ball, out_zone, hbp = d26['description'].isin(sp.BALL_DESC), d26['zone'] >= 11, d26['description'] == 'hit_by_pitch'
    r['사구 평균 점수 (raw)'] = float(rawc[hbp].mean())
    r['상위 1% 중 존 밖 볼 비율 (raw)'] = float((ball & out_zone)[top].mean())
    if k != 'LGBM v3':
        r['LGBM 과의 투구별 점수 상관 (2026)'] = float(np.corrcoef(d26[f'raw_{k}'], d26['raw_LGBM v3'])[0, 1])
        both = pd.concat([pt_means(d26, col), pt_means(d26, 's_LGBM v3')], axis=1, join='inner')
        r['LGBM 과의 투수×구종 평균 상관 (2026)'] = float(both.corr().iloc[0, 1])
    rows[k] = r
    log(f'구위 지표 {k} 완료')
stuff_cmp = pd.DataFrame(rows)
stuff_cmp.to_csv(os.path.join(OUT, 'stuff_metrics.csv'), encoding='utf-8-sig')
log('구위 지표\n' + stuff_cmp.round(4).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 5. 구위 산출 근거 : 운영 export_web_data.explain_contrib 를 그대로 호출 (2026 표본 2만 구)
# ---------------------------------------------------------------------------
class TreeShapV3:    # LGBM 은 정확한 TreeSHAP 으로 비교 (운영도 SHAP 캐시 사용)
    def __init__(self, m): self.m = m
    def explain(self, X, exact=False): return self.m.explain(X, exact=True)
    def method(self, exact=False): return 'shap'

s26 = fr[2026].sample(20000 if not SMOKE else 3000, random_state=SEED)
X26 = s26[v3.input_features]
hof = hn.HOFNetStuffModel(nets['A0'], prep, v3)
expl = {}
for k, model, sc in [('LGBM v3', TreeShapV3(v3), scalers['LGBM v3']), ('HOF-Net A0', hof, scalers['A0'])]:
    pred = (v3 if k == 'LGBM v3' else hof).predict(X26)
    score = sc.transform(pred)
    c, S0, method = ew.explain_contrib(model, sc, X26, pred, score, 2026, None)
    err = float(np.abs(S0 + c.sum(1) - score).max())
    expl[k] = {'c': c, 'S0': S0, 'method': method, 'score': score, 'additivity_err_points': err}
    log(f'근거 {k} : method {method}, S0 {S0:.2f}, 가법성 오차 {err:.2e}')
groups = [g for g, _ in ew.EXPLAIN_GROUPS]
agree = {}
for j, g in enumerate(groups):
    a, b = expl['LGBM v3']['c'][:, j], expl['HOF-Net A0']['c'][:, j]
    agree[g] = {'LGBM 평균 |기여| (점)': float(np.abs(a).mean()), 'HOF 평균 |기여| (점)': float(np.abs(b).mean()),
                '투구별 상관': float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else np.nan}
agree = pd.DataFrame(agree).T
ca, cb = expl['LGBM v3']['c'], expl['HOF-Net A0']['c']
top_up = float(np.mean(ca.argmax(1) == cb.argmax(1)))
top_dn = float(np.mean(ca.argmin(1) == cb.argmin(1)))
agree.to_csv(os.path.join(OUT, 'explain_agreement.csv'), encoding='utf-8-sig')
log('근거 일치도\n' + agree.round(3).to_string() + f'\n  최대 + 요인 일치 {top_up:.1%}, 최대 − 요인 일치 {top_dn:.1%}')

ex_rows = []
pick = {'HOF 고득점 FF': s26.assign(sc=expl['HOF-Net A0']['score'])[s26['pitch_type'].astype(str) == 'FF']['sc'].idxmax(),
        'HOF 고득점 SL': s26.assign(sc=expl['HOF-Net A0']['score'])[s26['pitch_type'].astype(str) == 'SL']['sc'].idxmax(),
        'HOF 저득점': s26.assign(sc=expl['HOF-Net A0']['score'])['sc'].idxmin()}
pos = {ix: i for i, ix in enumerate(s26.index)}
for name, ix in pick.items():
    i = pos[ix]
    for k in expl:
        ex_rows.append({'투구': name, '구종': str(s26.loc[ix, 'pitch_type']), '모델': k, '기준 점수': round(expl[k]['S0'], 1),
                        **{g: round(float(expl[k]['c'][i, j]), 1) for j, g in enumerate(groups)}, '구위': round(float(expl[k]['score'][i]), 1)})
examples = pd.DataFrame(ex_rows)
examples.to_csv(os.path.join(OUT, 'explain_examples.csv'), index=False, encoding='utf-8-sig')
log('근거 예시\n' + examples.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 6. 불확실성 · OOD · Go/No-Go
# ---------------------------------------------------------------------------
unc = {}
for k in ['A0', 'A1']:
    p26 = hn.predict(nets[k], T(fr[2026]))
    v = fr[2026]['release_speed'].to_numpy('float64', na_value=np.nan)
    hi, lo = v > 98, v <= 95
    u = p26['u_lat']
    lab = np.r_[np.ones(hi.sum()), np.zeros(lo.sum())]
    unc[k] = {'U_latent 평균 (2025 검증)': float(hof_va[k]['u_lat'].mean()), 'U_alea 평균 (2025 검증)': float(hof_va[k]['var'].mean()),
              'U_latent 비중 (2025 검증)': float(hof_va[k]['u_lat'].mean() / (hof_va[k]['u_lat'] + hof_va[k]['var']).mean()),
              'OOD 구속 >98 vs ≤95 AUROC (U_latent)': float(roc_auc_score(lab, np.r_[u[hi], u[lo]])) if hi.sum() and lo.sum() else np.nan,
              'OOD 구속 >98 / ≤95 U_latent 평균비': float(u[hi].mean() / u[lo].mean()) if hi.sum() else np.nan,
              '2026 / 2025 검증 U_latent 평균비': float(u.mean() / hof_va[k]['u_lat'].mean())}
unc = pd.DataFrame(unc)
unc.to_csv(os.path.join(OUT, 'uncertainty_ood.csv'), encoding='utf-8-sig')
log('불확실성 · OOD\n' + unc.round(5).to_string())

net = nets['A0']; net.eval()
with torch.no_grad():
    b = slice(0, 4096)
    o = net(va_t[0][b], va_t[1][b], va_t[2][b])
    Sig = o['s'].unsqueeze(2) * o['R'] * o['s'].unsqueeze(1)
    lam_min = float(torch.linalg.eigvalsh(Sig.double()).min())
    mean_c, var_lat, _ = None, None, None
    mean_cf, var_a, u_l = net.predictive(o)
    mc = {}
    for M in (16, 32, 64):
        torch.manual_seed(SEED)
        mm, vv = net.sample_mean(o, M)
        mc[M] = (float((mm - mean_cf).abs().max()), float(((vv - (var_a + u_l)).abs() / (var_a + u_l)).mean()))
h0 = hists['A0']
gonogo = {'Check 1 : Σ 최소 고윳값 (≥ −ε)': lam_min,
          'Check 2 : NaN 배치 / Cholesky 실패': f"{int(h0['nan_batches'].sum())} / {int(h0['chol_fail'].sum())}",
          'Check 3 : 마지막 epoch 학습 KL (붕괴 아님 > 0.01)': float(h0['train_kl'].iloc[-1]),
          'Check 4 : τ 최대 (폭주 아님)': float(h0['tau_max'].max()),
          **{f'Check 5 : MC M={M} 평균 최대 오차 / 분산 평균 상대오차': f'{a:.2e} / {r:.3f}' for M, (a, r) in mc.items()},
          'Check 6 : 대각(A1) 먼저 정상 동작 (A1 학습 완료, NaN 0)': f"{len(hists['A1'])} epoch, NaN {int(hists['A1']['nan_batches'].sum())}"}
summary = {'gonogo': gonogo, 'times_sec': times, 'epochs': {k: len(h) for k, h in hists.items()},
           'top_factor_agreement': {'up': top_up, 'down': top_dn},
           'explain': {k: {'S0': v['S0'], 'method': v['method'], 'additivity_err_points': v['additivity_err_points']} for k, v in expl.items()},
           'n': {'train': len(tr_rows), 'holdout': len(es_rows), 'valid': len(va_rows)}, 'smoke': SMOKE, 'total_sec': time.time() - T0}
json.dump(summary, open(os.path.join(OUT, 'summary.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
log('Go/No-Go', json.dumps(gonogo, ensure_ascii=False, indent=1))
log('끝')